# Regime-Conditional Calibration Check

This is the core research question: does Chronos-2's calibration failure
get worse specifically during high volatility periods?

We split our existing coverage results by regime label and compare.

In [1]:
import numpy as np

## Load everything: forecasts, actuals, and regime labels

In [2]:
sensex_forecasts = np.load("../data/forecasts.npy")
sensex_actuals = np.load("../data/actuals.npy")
sensex_regimes = np.load("../data/sensex_regime_labels.npy", allow_pickle=True)

nifty_forecasts = np.load("../data/nifty_forecasts.npy")
nifty_actuals = np.load("../data/nifty_actuals.npy")
nifty_regimes = np.load("../data/nifty_regime_labels.npy", allow_pickle=True)

sensex_forecasts.shape, nifty_forecasts.shape

((352, 30, 11), (403, 30, 11))

## Function to check coverage within one regime

Same coverage calculation as before, but now only using windows that
belong to a specific regime.

In [3]:
quantile_levels = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

def coverage_by_regime(forecasts, actuals, regimes, day):
    day_forecasts = forecasts[:, day, :]
    day_actuals = actuals[:, day]

    lower = day_forecasts[:, 1]
    upper = day_forecasts[:, 9]

    inside = (day_actuals >= lower) & (day_actuals <= upper)

    results = {}
    for regime in ["low_vol", "high_vol"]:
        mask = (regimes == regime)
        coverage = inside[mask].mean()
        count = mask.sum()
        results[regime] = (coverage, count)

    return results

## Sensex: coverage by regime, across horizons

In [4]:
print("SENSEX")
for day in [0, 1, 15, 29]:
    results = coverage_by_regime(sensex_forecasts, sensex_actuals, sensex_regimes, day)
    low_cov, low_n = results["low_vol"]
    high_cov, high_n = results["high_vol"]
    print(f"day {day}: low_vol = {round(low_cov*100,1)}% (n={low_n})   high_vol = {round(high_cov*100,1)}% (n={high_n})")

SENSEX
day 0: low_vol = 83.8% (n=197)   high_vol = 76.8% (n=155)
day 1: low_vol = 71.1% (n=197)   high_vol = 76.8% (n=155)
day 15: low_vol = 73.1% (n=197)   high_vol = 75.5% (n=155)
day 29: low_vol = 70.1% (n=197)   high_vol = 80.0% (n=155)


## Nifty: coverage by regime, across horizons

In [5]:
print("NIFTY")
for day in [0, 1, 15, 29]:
    results = coverage_by_regime(nifty_forecasts, nifty_actuals, nifty_regimes, day)
    low_cov, low_n = results["low_vol"]
    high_cov, high_n = results["high_vol"]
    print(f"day {day}: low_vol = {round(low_cov*100,1)}% (n={low_n})   high_vol = {round(high_cov*100,1)}% (n={high_n})")

NIFTY
day 0: low_vol = 85.7% (n=223)   high_vol = 74.4% (n=180)
day 1: low_vol = 76.7% (n=223)   high_vol = 73.9% (n=180)
day 15: low_vol = 69.1% (n=223)   high_vol = 72.8% (n=180)
day 29: low_vol = 71.7% (n=223)   high_vol = 80.0% (n=180)


## What to look for

If high_vol coverage is noticeably lower than low_vol coverage at the same
horizon, that supports our research question: calibration failure is worse
during volatile periods, not uniform across all market conditions.

If they're close to each other, that's also a real finding, it would mean
calibration failure is NOT regime-dependent, and is instead a general
property of the model regardless of market conditions.